# Proyecto de Clasificación con dataset KDD


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score

data_path = "../../../src/data/KDD/"

print("Cargando conjunto de entrenamiento (pequeño)...")
train = pd.read_csv(data_path + "train_1000000.csv", header=None)
validation = pd.read_csv(data_path + "validation.csv", header=None)
val_manifest = pd.read_csv(data_path + "validation.manifest.csv")

# Las primeras 41 columnas son features, la 41 (índice 41 si son 42 columnas en total) es el target.
# Asegurarse correctamente separando los datos:
X_train = train.iloc[:, :41]
y_train_raw = train.iloc[:, 41]

X_val = validation.iloc[:, :41]
y_val_raw = validation.iloc[:, 41]

# Conversión de etiquetas: 'normal.' -> 0, resto (ataques) -> 1
y_train = (y_train_raw != 'normal.').astype(int)
y_val = (y_val_raw != 'normal.').astype(int)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_val: {X_val.shape}")


Cargando conjunto de entrenamiento (pequeño)...
Dimensiones de X_train: (1000000, 41)
Dimensiones de X_val: (734765, 41)


## 2. Preprocesamiento
KDD contiene variables categóricas (Protocolo, Servicio, Estado). Utilizaremos `OneHotEncoder(handle_unknown='ignore')` para estas variables (índices 1, 2, 3), y `StandardScaler` para las numéricas.

In [3]:
categorical_cols = [1, 2, 3]
numerical_cols = [i for i in range(41) if i not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

print("Ajustando el preprocesador con los datos de entrenamiento...")
preprocessor.fit(X_train)

print("Transformando los conjuntos...")
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
print("Preprocesamiento terminado!")


Ajustando el preprocesador con los datos de entrenamiento...
Transformando los conjuntos...
Preprocesamiento terminado!


## 3. Entrenamiento y Evaluación
Probando los modelos (Decison Tree, Logistic Regression, Random Forest)

In [4]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
}

resultados = []
mejor_modelo = None
mejor_f1 = -1
nombre_mejor_modelo = ""

for nombre, modelo in modelos.items():
    print(f"\nEntrenando {nombre}...")
    start_time = time.time()
    modelo.fit(X_train_processed, y_train)
    tiempo_entrenamiento = time.time() - start_time
    
    # Evaluaciones
    y_pred = modelo.predict(X_val_processed)
    
    # Probabilidades para ROC AUC
    if hasattr(modelo, "predict_proba"):
        y_prob = modelo.predict_proba(X_val_processed)[:, 1]
        roc = roc_auc_score(y_val, y_prob)
    else:
        roc = roc_auc_score(y_val, y_pred)

    acc = accuracy_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred) # Positive label es 1 (Ataque)
    f1 = f1_score(y_val, y_pred)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    
    resultados.append({
        "Modelo": nombre,
        "Accuracy": acc,
        "Recall (Attack)": rec,
        "F1-Score (Attack)": f1,
        "ROC-AUC": roc,
        "Balanced Acc": bal_acc,
        "Validation Time": tiempo_entrenamiento,
        "Notas": "Parámetros base"
    })
    
    # KDD tiene más ataques que normal, basarse en F1-Score (Attack) es robusto:
    if f1 > mejor_f1:
        mejor_f1 = f1
        mejor_modelo = modelo
        nombre_mejor_modelo = nombre

df_resultados = pd.DataFrame(resultados)
print("\n--- RESUMEN DE RESULTADOS ---")
display(df_resultados)
print(f"\nMejor modelo seleccionado por F1-Score: {nombre_mejor_modelo}")



Entrenando Logistic Regression...

Entrenando Decision Tree...

Entrenando Random Forest...

--- RESUMEN DE RESULTADOS ---


,Modelo,Accuracy,Recall (Attack),F1-Score (Attack),ROC-AUC,Balanced Acc,Validation Time,Notas
0,Logistic Regression,0.994807,0.999309,0.996768,0.999913,0.987973,13.011407,Parámetros base
1,Decision Tree,0.999829,0.999796,0.999893,0.999930,0.999878,15.128463,Parámetros base
2,Random Forest,0.999865,0.999842,0.999916,0.999994,0.999900,45.872175,Parámetros base



Mejor modelo seleccionado por F1-Score: Random Forest


## 4. Generación de Predicciones

In [7]:
print(f"Generando archivo final de predicciones usando {nombre_mejor_modelo}...")

predicciones_final = mejor_modelo.predict(X_val_processed)

df_export = pd.DataFrame({
    'observation_id': val_manifest['observation_id'],
    'prediction': predicciones_final.astype(int)
})


df_export.to_csv("validation_predictions_KDD.csv", index=False)
print("Archivo validation_predictions_KDD.csv generado correctamente con estructura observation_id,prediction")

df_export.head()


Generando archivo final de predicciones usando Random Forest...
Archivo validation_predictions_KDD.csv generado correctamente con estructura observation_id,prediction


,observation_id,prediction
0,kddcup.data.gz:4,0
1,kddcup.data.gz:11,0
2,kddcup.data.gz:19,0
3,kddcup.data.gz:21,0
4,kddcup.data.gz:35,0
